Themes still needs functions for regularly updating symbol members in storage

In [2]:
%run "..\scripts\load_daily_variables.py"

Loading Variables: 100%|██████████| 27/27 [08:40<00:00, 19.27s/it]  


In [5]:
sys.path.insert(0, 'C:/Users/jdejo/Market_Data_Processing/market_data')
from io import StringIO

from api_keys import finviz_api_key

def theme_subtheme_membership(theme_subtheme):
    url = f"https://elite.finviz.com/export/screener?v=111&f={theme_subtheme}&auth={finviz_api_key}"
    response = requests.get(url)
    result = pd.read_csv(StringIO(response.text))
    return result


In [ ]:
import re

import requests
import pandas as pd
from bs4 import BeautifulSoup


URL = "https://finviz.com/screener"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/142.0.0.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


def get_options(select):
    return [
        {
            "value": option.get("value", "").strip(),
            "label": option.get_text(" ", strip=True),
        }
        for option in select.find_all("option")
    ]


# Examine every <select> on the page
selects = soup.find_all("select")

theme_select = None
subtheme_select = None

for select in selects:
    options = get_options(select)
    labels = [x["label"] for x in options]

    # Theme dropdown should contain this exact theme.
    if "Artificial Intelligence" in labels:
        theme_select = select

    # Sub-theme dropdown contains labels such as:
    # "Agriculture - Alternative Proteins"
    if any(
        label.startswith("Agriculture - Alternative Proteins")
        for label in labels
    ):
        subtheme_select = select


if theme_select is None:
    raise RuntimeError("Could not identify the FINVIZ Theme dropdown.")

if subtheme_select is None:
    raise RuntimeError("Could not identify the FINVIZ Sub-theme dropdown.")


# -----------------------------
# Extract themes
# -----------------------------

theme_options = [
    x for x in get_options(theme_select)
    if x["label"].lower() != "any" and x["label"]
]

_themes = pd.DataFrame(
    {
        "theme": x["label"],
        "theme_value": x["value"],
    }
    for x in theme_options
)


# -----------------------------
# Extract sub-themes
# -----------------------------

subtheme_options = [
    x for x in get_options(subtheme_select)
    if x["label"].lower() != "any" and x["label"]
]

# FINVIZ abbreviates many parent names in the sub-theme dropdown.
# Map those prefixes back to the canonical names in the theme dropdown.
subtheme_prefix_aliases = {
    "AI": "Artificial Intelligence",
    "Agriculture": "Agriculture & FoodTech",
    "Automation": "Industrial Automation",
    "Autonomous": "Autonomous Systems",
    "Blockchain": "Crypto & Blockchain",
    "Cloud": "Cloud Computing",
    "Comm Agri": "Commodities - Agriculture",
    "Comm Energy": "Commodities - Energy",
    "Comm Metals": "Commodities - Metals",
    "Consumer": "Consumer Goods",
    "Defense": "Defense & Aerospace",
    "Education": "Education Technology",
    "Energy Base": "Energy - Traditional",
    "Energy Clean": "Energy - Renewable",
    "Entertainment": "Digital Entertainment",
    "Environmental": "Environmental Sustainability",
    "EVs": "Electric Vehicles",
    "Healthcare": "Healthcare & Biotech",
    "IoT": "Internet of Things",
    "Longevity": "Aging Population & Longevity",
    "NanoTech": "Nanotechnology",
    "Nutrition": "Healthy Food & Nutrition",
    "Quantum": "Quantum Computing",
    "Real Estate": "Real Estate & REITs",
    "Semis": "Semiconductors",
    "Social": "Social Media",
    "Space": "Space Tech",
    "Telecom": "Telecommunications",
    "Transportation": "Transportation & Logistics",
    "V/A Reality": "Virtual & Augmented Reality",
}

theme_names = set(_themes["theme"])
prefix_to_theme = {theme: theme for theme in theme_names}
prefix_to_theme.update(subtheme_prefix_aliases)

unknown_alias_targets = set(subtheme_prefix_aliases.values()) - theme_names
if unknown_alias_targets:
    raise RuntimeError(
        "FINVIZ theme names changed; alias targets were not found: "
        + ", ".join(sorted(unknown_alias_targets))
    )

rows = []
unmatched_labels = []

for option in subtheme_options:
    full_label = option["label"]
    parent_theme = None
    subtheme_name = full_label

    # Handle the non-hierarchical Custom option separately.
    if full_label in theme_names:
        parent_theme = full_label
    else:
        match = re.match(r"^(.*?)\s+-\s+(.*)$", full_label)
        if match:
            parent_prefix = match.group(1).strip()
            parent_theme = prefix_to_theme.get(parent_prefix)
            subtheme_name = match.group(2).strip()

    if parent_theme is None:
        unmatched_labels.append(full_label)

    rows.append(
        {
            "theme": parent_theme,
            "subtheme": subtheme_name,
            "subtheme_full_label": full_label,
            "subtheme_value": option["value"],
        }
    )

if unmatched_labels:
    raise RuntimeError(
        "Could not map these FINVIZ sub-themes to parent themes: "
        + ", ".join(unmatched_labels)
    )

subthemes = pd.DataFrame(rows)

# Add theme's underlying FINVIZ value
taxonomy = subthemes.merge(
    _themes,
    how="left",
    on="theme",
    validate="many_to_one",
)

taxonomy = taxonomy[
    [
        "theme",
        "theme_value",
        "subtheme",
        "subtheme_value",
        "subtheme_full_label",
    ]
].sort_values(
    ["theme", "subtheme"]
)

taxonomy.to_csv(
    "finviz_theme_taxonomy.csv",
    index=False
)

print(f"Themes:    {len(_themes):,}")
print(f"Sub-themes:{len(subthemes):,}")
print()

print(taxonomy.to_string(index=False))

Themes:    41
Sub-themes:269

                       theme                 theme_value                                        subtheme               subtheme_value                                              subtheme_full_label
Aging Population & Longevity    agingpopulationlongevity           Age-Related Pharmaceuticals & Biotech         longevityagingpharma                Longevity - Age-Related Pharmaceuticals & Biotech
Aging Population & Longevity    agingpopulationlongevity                    Healthcare & Medical Devices          longevityhealthcare                         Longevity - Healthcare & Medical Devices
Aging Population & Longevity    agingpopulationlongevity                       Healthy Aging & Nutrition        longevityhealthyaging                            Longevity - Healthy Aging & Nutrition
Aging Population & Longevity    agingpopulationlongevity                   Senior Living & Assisted Care        longevityseniorliving                        Longevity - Senio

In [37]:
import requests
import pandas as pd
from bs4 import BeautifulSoup


URL = "https://finviz.com/screener"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/142.0.0.0 Safari/537.36"
    )
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


def get_options(select):
    return [
        {
            "value": option.get("value", "").strip(),
            "label": option.get_text(" ", strip=True),
        }
        for option in select.find_all("option")
    ]


# Find the Theme dropdown
theme_select = None

for select in soup.find_all("select"):
    options = get_options(select)
    labels = [x["label"] for x in options]

    if "Artificial Intelligence" in labels:
        theme_select = select
        break


if theme_select is None:
    raise RuntimeError("Could not identify FINVIZ Theme dropdown.")


# Extract theme names and their underlying FINVIZ values
rows = []

for option in get_options(theme_select):

    label = option["label"]
    value = option["value"]

    if not label or label.lower() == "any":
        continue

    rows.append({
        "theme": label,
        "theme_value": value,
        "finviz_filter": f"theme_{value}",
    })


themes = pd.DataFrame(rows)

themes = themes.sort_values("theme").reset_index(drop=True)

themes.to_csv(
    "finviz_themes.csv",
    index=False
)

print(f"Themes found: {len(themes)}")
print()
print(themes.to_string(index=False))

Themes found: 41

                       theme                 theme_value                     finviz_filter
Aging Population & Longevity    agingpopulationlongevity    theme_agingpopulationlongevity
      Agriculture & FoodTech         agriculturefoodtech         theme_agriculturefoodtech
     Artificial Intelligence      artificialintelligence      theme_artificialintelligence
          Autonomous Systems           autonomoussystems           theme_autonomoussystems
                    Big Data                     bigdata                     theme_bigdata
                  Biometrics                  biometrics                  theme_biometrics
             Cloud Computing              cloudcomputing              theme_cloudcomputing
   Commodities - Agriculture      commoditiesagriculture      theme_commoditiesagriculture
        Commodities - Energy           commoditiesenergy           theme_commoditiesenergy
        Commodities - Metals           commoditiesmetals           theme

In [42]:
finviz_themes = themes.merge(taxonomy, how='outer', on=['theme', 'theme_value'])

In [45]:
finviz_themes["finviz_subtheme_filter"] = (
    finviz_themes["finviz_subtheme_filter"]
    .str.replace(r"^(?!subtheme_)", "subtheme_", regex=True)
)

In [46]:
finviz_themes.head()

,theme,theme_value,finviz_theme_filter,subtheme,finviz_subtheme_filter,subtheme_full_label
0,Aging Population & Longevity,agingpopulationlongevity,theme_agingpopulationlongevity,Age-Related Pharmaceuticals & Biotech,subtheme_longevityagingpharma,Longevity - Age-Related Pharmaceuticals & Biotech
1,Aging Population & Longevity,agingpopulationlongevity,theme_agingpopulationlongevity,Healthcare & Medical Devices,subtheme_longevityhealthcare,Longevity - Healthcare & Medical Devices
2,Aging Population & Longevity,agingpopulationlongevity,theme_agingpopulationlongevity,Healthy Aging & Nutrition,subtheme_longevityhealthyaging,Longevity - Healthy Aging & Nutrition
3,Aging Population & Longevity,agingpopulationlongevity,theme_agingpopulationlongevity,Senior Living & Assisted Care,subtheme_longevityseniorliving,Longevity - Senior Living & Assisted Care
4,Agriculture & FoodTech,agriculturefoodtech,theme_agriculturefoodtech,Agri-Food Processing & Distribution,subtheme_agricultureprocessing,Agriculture - Agri-Food Processing & Distribution


In [52]:
themes_subthemes = {}
for theme in finviz_themes.finviz_theme_filter.unique():
    themes_subthemes[theme] = theme_subtheme_membership(theme).Ticker.to_list()
    time.sleep(5)

for subtheme in finviz_themes.finviz_subtheme_filter.unique():
    themes_subthemes[subtheme] = theme_subtheme_membership(subtheme).Ticker.to_list()
    time.sleep(5)

In [59]:
themes_df = pd.DataFrame(
    [
        {"symbols": symbol, "theme_subtheme": theme_subtheme}
        for theme_subtheme, symbols in themes_subthemes.items()
        for symbol in symbols
    ]
)

themes_df.head()

,symbols,theme_subtheme
0,ABT,theme_agingpopulationlongevity
1,BAX,theme_agingpopulationlongevity
2,BIIB,theme_agingpopulationlongevity
3,BKD,theme_agingpopulationlongevity
4,BMY,theme_agingpopulationlongevity


In [64]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL as SQLAlchemyURL

from api_keys import database_password


stocks_engine = create_engine(
    SQLAlchemyURL.create(
        "mysql+pymysql",
        username="root",
        password=database_password,
        host="127.0.0.1",
        port=3306,
        database="stocks",
    ),
    pool_pre_ping=True,
)

sql_finviz_themes = (
    finviz_themes[
        [
            "theme",
            "theme_value",
            "finviz_theme_filter",
            "subtheme",
            "finviz_subtheme_filter",
            "subtheme_full_label",
        ]
    ]
)

with stocks_engine.begin() as connection:
    sql_finviz_themes.to_sql(
        name="finviz_themes",
        con=connection,
        schema="stocks",
        if_exists="append",
        index=False,
        method="multi",
    )

print(f"Stored {len(sql_finviz_themes):,} rows in stocks.finviz_themes.")

Stored 269 rows in stocks.finviz_themes.


In [90]:
valid_mask = themes_df["theme_subtheme"].str.match(
    r"^(theme|subtheme)_.+$",
    na=False,
)

themes_df_to_store = themes_df.loc[valid_mask].copy()

print("Original rows:", len(themes_df))
print("Rows to store:", len(themes_df_to_store))
print("Rows excluded:", (~valid_mask).sum())

themes_df.loc[~valid_mask, "theme_subtheme"].value_counts(dropna=False)

Original rows: 27176
Rows to store: 4038
Rows excluded: 23138


theme_subtheme
theme_       11569
subtheme_    11569
Name: count, dtype: int64

In [92]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL as SQLAlchemyURL

from api_keys import database_password


stocks_engine = create_engine(
    SQLAlchemyURL.create(
        "mysql+pymysql",
        username="root",
        password=database_password,
        host="127.0.0.1",
        port=3306,
        database="stocks",
    ),
    pool_pre_ping=True,
)

sql_symbol_themes = themes_df_to_store.rename(columns={"symbols": "symbol"})[
    ["symbol", "theme_subtheme"]
]

with stocks_engine.begin() as connection:
    sql_symbol_themes.to_sql(
        name="finviz_symbol_themes",
        con=connection,
        schema="stocks",
        if_exists="append",
        index=False,
        method="multi",
    )

print(
    f"Stored {len(sql_symbol_themes):,} rows in stocks.finviz_symbol_themes."
)

Stored 4,038 rows in stocks.finviz_symbol_themes.


In [7]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL as SQLAlchemyURL

stocks_engine = create_engine(
    SQLAlchemyURL.create(
        "mysql+pymysql",
        username="root",
        password=database_password,
        host="127.0.0.1",
        port=3306,
        database="stocks",
    ),
    pool_pre_ping=True,
)

db_finviz_themes = pd.read_sql_table(
    table_name="finviz_themes",
    con=stocks_engine,
    schema="stocks",
)

db_finviz_symbol_themes = pd.read_sql_table(
    table_name="finviz_symbol_themes",
    con=stocks_engine,
    schema="stocks",
)

print(f"Loaded {len(db_finviz_themes):,} rows from stocks.finviz_themes.")
print(
    f"Loaded {len(db_finviz_symbol_themes):,} rows "
    "from stocks.finviz_symbol_themes."
)

Loaded 269 rows from stocks.finviz_themes.
Loaded 4,038 rows from stocks.finviz_symbol_themes.
